# 📝 Homework Day 3: Evaluation & Optimization
## Agentic RAG: From Zero to Hero

**Due:** 1 week

## 🎓 Fill in student information

In [ ]:
# ─── Fill in your information ───
STUDENT_NAME = ''   # e.g. 'Somchai Jaidee'
STUDENT_ID   = ''   # e.g. '6512345678'
PHONE        = ''   # e.g. '081-234-5678'
LINE_ID      = ''   # e.g. 'somchai.j'

# ─── Validation (do not edit) ───
assert len(STUDENT_ID) >= 5, '❌ Please enter your student ID!'
assert len(STUDENT_NAME) >= 2, '❌ Please enter your full name!'

print(f'✅ Full name: {STUDENT_NAME}')
print(f'✅ Student ID: {STUDENT_ID}')
print(f'📱 Phone number: {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')

---
## 📦 Install Dependencies

In [ ]:
%%time
!pip install -q ragas datasets google-adk google-genai \
    sentence-transformers qdrant-client langchain-text-splitters rank_bm25

import os, json, hashlib, random, asyncio, warnings
import numpy as np
warnings.filterwarnings('ignore')

try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
except:
    os.environ['GOOGLE_API_KEY'] = input('🔑 API Key: ')
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'False'

from google import genai
client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])
print('✅ Setup complete')

## 📄 Create a personalized dataset (do not edit)

In [ ]:
%%time
# ─── Anti-Cheat: Generate data specific to the student ID ───
seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (10**9)
rng = random.Random(seed)

all_topics = {
    'healthcare': [
        'Siriraj Hospital uses AI to analyze chest X-ray images with 95% accuracy, reducing diagnosis time from 30 minutes to 5 minutes',
        'NLP analyzes electronic medical records to help doctors summarize patient history faster, reducing time from 15 minutes to 2 minutes',
        'An AI system helps screen for diabetes from retinal images, using Deep Learning to detect it from the early stages',
    ],
    'banking': [
        'Kasikornbank uses RAG to answer customer questions, reducing Call Center workload by 40% and enabling 24-hour service',
        'A fraud detection system uses Machine Learning to analyze transactions in real time, reducing fraud by 60%',
        'AI analyzes credit risk using alternative data such as phone bill history and utility bills',
    ],
    'education': [
        'Chulalongkorn University built a RAG system for question answering from 500 textbooks, allowing students to learn independently 24 hours a day',
        'An Intelligent Tutoring System adjusts content to the learner’s level using an adaptive learning algorithm',
        'AI helps grade Thai essay exams, reducing grading time by 70% with 88% accuracy compared to human instructors',
    ],
    'agriculture': [
        'Smart Farming uses AI to analyze drone images to detect 8 types of plant diseases with 92% accuracy',
        'A rice yield forecasting system uses satellite data, IoT sensors, and weather data with 85% accuracy',
        'AI analyzes agricultural product prices from news and social media to help farmers make selling decisions',
    ],
    'logistics': [
        'An AI system improves transportation routes, reducing fuel costs by 25% using Graph Neural Network',
        'An intelligent warehouse uses AGV robots and Computer Vision to automatically organize products',
        'A logistics customer service chatbot uses Thai NLP to answer parcel status and shipping price questions in real time',
    ],
}

topics = rng.sample(list(all_topics.keys()), 4)
my_data = {}
for t in topics:
    my_data[t] = rng.sample(all_topics[t], 2)

print(f'📋 Your topics ({STUDENT_ID}):')
for t, texts in my_data.items():
    print(f'  📌 {t}: {len(texts)} texts')
    for tx in texts:
        print(f'     → {tx[:50]}...')

---
## 🎯 Step 1: Build a RAG Pipeline (3 points)

Create a pipeline from `my_data`:
1. Chunk using RecursiveCharacterTextSplitter
2. Embed using multilingual-e5-large
3. Upsert into Qdrant
4. Create functions `search_qdrant(query)` + `rag_answer(question)`

In [ ]:
%%time
# ─── Step 1: Build a RAG Pipeline ───
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 💡 Hint:
#   1. embed_model = SentenceTransformer('intfloat/multilingual-e5-large')
#   2. splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
#   3. Loop through my_data → chunk → embed → store in a list
#   4. qdrant = QdrantClient(':memory:') → create_collection → upsert
#   5. Create search_qdrant(query, top_k=3) → return [{text, source, score}]
#   6. Create rag_answer(question) → search + Gemini → return answer

# embed_model = SentenceTransformer('intfloat/multilingual-e5-large')
# splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
#
# all_chunks, all_sources = [], []
# for src, texts in my_data.items():
#     for text in texts:
#         for chunk in splitter.split_text(text):
#             all_chunks.append(chunk)
#             all_sources.append(src)
#
# def search_qdrant(query, top_k=3):
#     ...
#
# def rag_answer(question, top_k=3):
#     ...


---
## 🎯 Step 2: Evaluate RAG Quality (2.5 points)

1. Create at least 5 evaluation questions (from your own data)
2. Evaluate using **LLM-as-Judge** with a score from 1-5
3. Show the average score

In [ ]:
%%time
# ─── Step 2: Evaluate Quality ───

# 💡 Hint:
#   1. Create eval_questions = ['question1', 'question2', ...] with at least 5 questions
#   2. Loop through rag_answer() and collect answers
#   3. Use llm_judge() to score each question
#   4. Calculate the average

def llm_judge(question, answer):
    prompt = f'''Score from 1-5:
Q: {question}
A: {answer}
Respond in JSON: {{"score": 1-5, "reason": "..."}}'''
    resp = client.models.generate_content(model='gemini-2.5-flash', contents=prompt,
        config=genai.types.GenerateContentConfig(temperature=0.1, response_mime_type='application/json'))
    return json.loads(resp.text)

# eval_questions = [
#     '...',
#     '...',
# ]
# scores = []
# for q in eval_questions:
#     ans, _ = rag_answer(q)
#     verdict = llm_judge(q, ans)
#     scores.append(verdict['score'])
#     print(f"  {'⭐'*verdict['score']} {q[:30]}...")
# print(f'Average: {sum(scores)/len(scores):.1f}/5.0')


---
## 🎯 Step 3: Improve the Pipeline (2.5 points)

1. Make at least **2 improvements** (for example chunk_size + prompt)
2. Measure **Before/After** using LLM-as-Judge
3. Explain **why** it improved or did not improve

In [ ]:
%%time
# ─── Step 3: Improve ───

# 💡 Hint:
#   1. Save baseline_avg from Step 2
#   2. Adjust chunk_size, for example 100→300 → re-embed → re-search → evaluate again
#   3. Adjust the prompt in rag_answer, for example add "Answer concisely and cite the data"
#   4. Compare before vs after

# baseline_avg = ...  # from Step 2
# 
# # Improvement 1: change chunk_size
# # Improvement 2: change prompt
# 
# improved_avg = ...
# print(f'Before: {baseline_avg:.1f} → After: {improved_avg:.1f}')
# print(f'Reason: ...')


---
## 🎯 Step 4: Build an Agent + Test (2 points)

1. Build an Agent using Google ADK that uses a Tool to search from RAG
2. Write at least **5 test cases**
3. Measure the pass rate

In [ ]:
%%time
# ─── Step 4: Agent + Testing ───
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# 💡 Hint:
#   1. Create a tool that calls search_qdrant()
#      def search_knowledge(query: str) -> dict:
#          results = search_qdrant(query, top_k=3)
#          return {'results': results}
#
#   2. Create an Agent
#      my_agent = Agent(name='hw3', model='gemini-2.5-flash',
#          instruction='...', tools=[search_knowledge])
#
#   3. Write test cases
#      test_cases = [
#          {'input': '...', 'expected_tool': 'search_knowledge'},
#          {'input': '...', 'expected_tool': None},
#      ]
#
#   4. Run tests and measure the pass rate


---
## 📊 Grading Criteria

| Step | Points | Criteria |
|---------|:-----:|------|
| 1. RAG Pipeline | 3 | Pipeline works, can search, rag_answer can answer |
| 2. Quality Evaluation | 2.5 | eval questions ≥5, has LLM-as-Judge scores |
| 3. Improvement | 2.5 | ≥2 improvements, has Before/After, explains reasons |
| 4. Agent + Test | 2 | Agent works, test cases ≥5, has pass rate |
| **Total** | **10** | |

## ✅ Check your answers

In [ ]:
# ─── Check before submission (do not edit) ───
print('═' * 50)
print(f'📋 Day 3 Homework Summary')
print(f'═' * 50)
print(f'Name: {STUDENT_NAME}')
print(f'ID: {STUDENT_ID}')
print(f'Topics: {list(my_data.keys())}')

checks = {
    'RAG Pipeline': 'search_qdrant' in dir() or 'search_qdrant' in globals(),
    'rag_answer': 'rag_answer' in dir() or 'rag_answer' in globals(),
    'llm_judge': 'llm_judge' in dir() or 'llm_judge' in globals(),
}

for name, ok in checks.items():
    print(f"  {'✅' if ok else '❌'} {name}")

if all(checks.values()):
    print('\n🎉 Ready to submit!')
else:
    print('\n⚠️ Not complete yet. Please finish the missing parts.')